# NYC Taxi Data Pipeline: Data Processing
## Teresa Ferrill
#### October 8, 2026

This notebook prepares January 2025 yellow-taxi records for analytics.  It separates records by pickup month, adds data quality flags, and integrates pickup and drop-off location descriptions.  

Raw files remain unchanged. Quality flags identify records requiring review; they do not automatically remove records from every analysis.

Run this notebook from the notebooks folder.    

### Load the source files

In [1]:
from pathlib import Path
import pandas as pd

raw_dir = Path("../data/raw")

trips = pd.read_parquet(
    raw_dir / "yellow_tripdata_2025-01.parquet"
)
zones = pd.read_csv(raw_dir / "taxi_zone_lookup.csv")

# Preserve each record's position within its source file.
trips["source_file"] = "yellow_tripdata_2025-01.parquet"
trips["source_row"] = range(len(trips))

print(f"Raw trip records: {len(trips):,}")
print(f"Lookup records: {len(zones):,}")

# Prevent lookup joins from multiplying records.
assert zones["LocationID"].notna().all(), "Missing lookup IDs"
assert zones["LocationID"].is_unique, "Duplicate lookup IDs"

Raw trip records: 3,475,226
Lookup records: 265


### 1. Separate Records by Reporting Period

January records have pickup timestamps on or after January 1, 2025,
and before February 1, 2025. Drop-offs may occur in February.

Records outside this period, including missing pickup timestamps,
are retained separately for review.

In [2]:
pickup = trips["tpep_pickup_datetime"]

in_january = (
    pickup.ge(pd.Timestamp("2025-01-01"))
    & pickup.lt(pd.Timestamp("2025-02-01"))
)

period_review = trips.loc[~in_january].copy()
period_review["review_reason"] = "Pickup outside January 2025"
period_review.loc[
    period_review["tpep_pickup_datetime"].isna(),
    "review_reason"
] = "Missing pickup timestamp"

january_trips = trips.loc[in_january].copy()

assert len(january_trips) + len(period_review) == len(trips)

print(f"January records: {len(january_trips):,}")
print(f"Reporting-period review records: {len(period_review):,}")

display(period_review.head())

January records: 3,475,204
Reporting-period review records: 22


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,source_file,source_row,review_reason
605,2,2024-12-31 23:30:03,2024-12-31 23:43:02,1.0,3.00,1.0,N,246,13,1,...,5.32,0.0,1.0,26.62,2.5,0.0,0.0,yellow_tripdata_2025-01.parquet,605,Pickup outside January 2025
687,2,2024-12-31 23:31:38,2024-12-31 23:41:48,1.0,1.03,1.0,N,43,140,2,...,0.00,0.0,1.0,15.70,2.5,0.0,0.0,yellow_tripdata_2025-01.parquet,687,Pickup outside January 2025
688,2,2024-12-31 23:46:38,2025-01-01 00:03:03,1.0,3.95,1.0,N,229,24,2,...,0.00,0.0,1.0,24.10,2.5,0.0,0.0,yellow_tripdata_2025-01.parquet,688,Pickup outside January 2025
861,2,2024-12-31 23:56:19,2025-01-01 00:11:19,6.0,2.28,1.0,N,68,107,1,...,3.98,0.0,1.0,23.88,2.5,0.0,0.0,yellow_tripdata_2025-01.parquet,861,Pickup outside January 2025
1108,2,2024-12-31 23:55:37,2025-01-01 00:01:26,1.0,1.12,1.0,N,56,56,2,...,0.00,0.0,1.0,10.40,0.0,0.0,0.0,yellow_tripdata_2025-01.parquet,1108,Pickup outside January 2025


### 2. Add Data Quality Flags

Flags identify nonpositive durations and distances, negative fares, missing supplementary fields, and unmatched location IDs.  

Missing durations and distances are flagged separately. Upper-limit rules for extreme values have not yet been established, so these flags do not certify that a record is suitable for every calculation.  

In [3]:
january_trips["duration_minutes"] = (
    january_trips["tpep_dropoff_datetime"]
    - january_trips["tpep_pickup_datetime"]
).dt.total_seconds() / 60

january_trips["flag_missing_duration"] = (
    january_trips["duration_minutes"].isna()
)
january_trips["flag_nonpositive_duration"] = (
    january_trips["duration_minutes"].le(0)
)

january_trips["flag_missing_distance"] = (
    january_trips["trip_distance"].isna()
)
january_trips["flag_nonpositive_distance"] = (
    january_trips["trip_distance"].le(0)
)

january_trips["flag_negative_fare"] = (
    january_trips["fare_amount"].lt(0)
)

supplementary_fields = [
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "Airport_fee"
]

for column in supplementary_fields:
    january_trips[f"flag_missing_{column}"] = (
        january_trips[column].isna()
    )

valid_ids = zones["LocationID"]

january_trips["flag_unmatched_pickup"] = (
    ~january_trips["PULocationID"].isin(valid_ids)
)
january_trips["flag_unmatched_dropoff"] = (
    ~january_trips["DOLocationID"].isin(valid_ids)
)

flag_columns = [
    column for column in january_trips.columns
    if column.startswith("flag_")
]

flag_summary = pd.DataFrame({
    "flagged_count": january_trips[flag_columns].sum(),
    "flagged_percent": (
        january_trips[flag_columns].mean() * 100
    ).round(4)
})

display(flag_summary)

,flagged_count,flagged_percent
flag_missing_duration,0,0.0000
flag_nonpositive_duration,2051,0.0590
flag_missing_distance,0,0.0000
flag_nonpositive_distance,90893,2.6155
flag_negative_fare,144118,4.1470
flag_missing_passenger_count,540149,15.5429
flag_missing_RatecodeID,540149,15.5429
flag_missing_store_and_fwd_flag,540149,15.5429
flag_missing_congestion_surcharge,540149,15.5429
flag_missing_Airport_fee,540149,15.5429


### 3. Integrate Location Descriptions

Left joins add pickup and drop-off boroughs and zone descriptions.
Many-to-one validation ensures each trip matches at most one lookup
record. Row counts must remain unchanged.

In [4]:
pickup_lookup = zones[
    ["LocationID", "Borough", "Zone"]
].rename(columns={
    "LocationID": "PULocationID",
    "Borough": "pickup_borough",
    "Zone": "pickup_zone"
})

dropoff_lookup = zones[
    ["LocationID", "Borough", "Zone"]
].rename(columns={
    "LocationID": "DOLocationID",
    "Borough": "dropoff_borough",
    "Zone": "dropoff_zone"
})

before_join = len(january_trips)

january_trips = january_trips.merge(
    pickup_lookup,
    on="PULocationID",
    how="left",
    validate="many_to_one"
).merge(
    dropoff_lookup,
    on="DOLocationID",
    how="left",
    validate="many_to_one"
)

assert len(january_trips) == before_join
assert january_trips["source_row"].is_unique

print(f"Records after location joins: {len(january_trips):,}")

display(january_trips[[
    "source_row",
    "PULocationID",
    "pickup_borough",
    "pickup_zone",
    "DOLocationID",
    "dropoff_borough",
    "dropoff_zone",
    "duration_minutes"
]].head())

Records after location joins: 3,475,204


,source_row,PULocationID,pickup_borough,pickup_zone,DOLocationID,dropoff_borough,dropoff_zone,duration_minutes
0,0,229,Manhattan,Sutton Place/Turtle Bay North,237,Manhattan,Upper East Side South,8.350000
1,1,236,Manhattan,Upper East Side North,237,Manhattan,Upper East Side South,2.550000
2,2,141,Manhattan,Lenox Hill West,141,Manhattan,Lenox Hill West,1.950000
3,3,244,Manhattan,Washington Heights South,244,Manhattan,Washington Heights South,5.566667
4,4,244,Manhattan,Washington Heights South,116,Manhattan,Hamilton Heights,3.533333


### Processing Results

The notebook retained 3,475,204 records with January 2025 pickup
timestamps and separated 22 records for reporting-period review.

Data quality flags were added without removing flagged January records.
Pickup and drop-off location descriptions were integrated through
validated many-to-one joins. The joins preserved the January row count
and unique source-row references.

Flag percentages use January records as the denominator. Raw files
remain unchanged. Extreme values still require review before use in
distance, duration, speed, or fare summaries.

The processed and review datasets currently exist in memory; export
will be implemented in the next step.

### 4. Export Processed Data and Reports

Export January records with their quality flags and location descriptions
to Parquet. Save reporting-period review records separately and export
the flag summary to CSV.

Raw files remain unchanged. Rerunning this section replaces these outputs.
The January dataset retains flagged records and is not fully cleaned.

In [5]:
processed_dir = Path("../data/processed")
review_dir = Path("../data/review")
report_dir = Path("../reports")

for folder in [processed_dir, review_dir, report_dir]:
    folder.mkdir(parents=True, exist_ok=True)

january_path = processed_dir / "yellow_trips_2025-01_processed.parquet"
review_path = review_dir / "yellow_trips_2025-01_period_review.parquet"
summary_path = report_dir / "processing_flag_summary_2025-01.csv"

# Export datasets without the pandas index.
january_trips.to_parquet(january_path, index=False)
period_review.to_parquet(review_path, index=False)

# Preserve flag names as an explicit CSV column.
flag_summary.rename_axis("flag").reset_index().to_csv(
    summary_path, index=False
)

# Verify saved row counts without reloading every column.
import pyarrow.parquet as pq

assert pq.ParquetFile(january_path).metadata.num_rows == len(january_trips)
assert pq.ParquetFile(review_path).metadata.num_rows == len(period_review)

print(f"Processed January records exported: {len(january_trips):,}")
print(f"Reporting-period review records exported: {len(period_review):,}")

for path in [january_path, review_path, summary_path]:
    print(f"\nSaved: {path.resolve()}")
    print(f"Size: {path.stat().st_size / (1024 ** 2):.2f} MB")

Processed January records exported: 3,475,204
Reporting-period review records exported: 22

Saved: C:\Users\ferri\anaconda_projects\DSC680\Week_09\NYC_Taxi\data\processed\yellow_trips_2025-01_processed.parquet
Size: 96.95 MB

Saved: C:\Users\ferri\anaconda_projects\DSC680\Week_09\NYC_Taxi\data\review\yellow_trips_2025-01_period_review.parquet
Size: 0.01 MB

Saved: C:\Users\ferri\anaconda_projects\DSC680\Week_09\NYC_Taxi\reports\processing_flag_summary_2025-01.csv
Size: 0.00 MB
